# Análisis de Errores
¿Qué churners no detectamos (FN)? ¿Qué renewals se alarman en falso (FP)?
Ayuda a entender los límites del modelo y guiar mejoras.

In [ ]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.models.error_analysis_08 import (
    load_labeled_test, classify_predictions, analyze_errors,
    plot_confusion_heatmap, plot_error_distributions, plot_score_by_result,
)
from src.models.train_06 import best_f1_threshold, MODELS_DIR, FEATURE_COLS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Cargar datos y modelo

In [ ]:
df_test, X_test, y_test = load_labeled_test()
artifact = joblib.load(MODELS_DIR / 'best_model.joblib')
model = artifact['model']
print(f'Modelo: {artifact["name"]} | test: {len(df_test):,} usuarios')

proba = model.predict_proba(X_test)[:, 1]
threshold = best_f1_threshold(y_test, proba)
preds = (proba >= threshold).astype(int)
print(f'Umbral óptimo F1: {threshold:.3f}')

## 2. Matriz de confusión

In [ ]:
plot_confusion_heatmap(y_test, preds)

## 3. Clasificación de predicciones (TP / FP / FN / TN)

In [ ]:
df_labeled = classify_predictions(df_test, proba, y_test, threshold)
analyze_errors(df_labeled)
df_labeled['result'].value_counts()

## 4. Distribuciones por tipo de error

In [ ]:
plot_error_distributions(df_labeled)

## 5. Scores de FN por características

In [ ]:
plot_score_by_result(df_labeled, threshold)

## 6. Perfil medio por resultado

In [ ]:
profile_cols = [
    'last_is_cancel', 'last_is_auto_renew', 'n_transactions',
    'days_since_last', 'listening_trend', 'tenure_days',
    'ever_canceled', 'price_trend',
]
profile = df_labeled.groupby('result')[profile_cols].mean().T
profile.style.background_gradient(axis=1, cmap='RdYlGn')